# Fase 3.7 — expansão interna, execução manual no Colab

A 3.7 acrescenta amostras de dois novos shards do Community Forensics–Small aos dados da 3.5. O dataset continua o mesmo: esta versão testa maior cobertura interna, sem afirmar independência de fonte. Arquitetura, checkpoint inicial da 3.3, aumentos, amostrador e limite de 12 épocas permanecem iguais aos da 3.5. Os diretórios 3.5 e 3.6 são preservados.

Execute as células 1–5 para preparar o runtime. A 6 faz o piloto, que confere classes e grupos, mas não seleciona dados. A 7 baixa e congela a seleção e as divisões. A 8 copia os dados para /content. A 9 é o smoke test: verifica hashes, checkpoint, forward e gravação, sem treinar. A 10 inicia ou retoma o treino manualmente. A 11 lê os artefatos internos. Após o treino, a 12 copia Mirage, a 13 congela checkpoint, temperatura e limiar, a 14 avalia e a 15 lê o resultado. Mirage já foi observado: serve somente para comparação, sem orientar mudanças nesta versão.

Conecte uma GPU. Não use “Run All”: o treino completo ocorre somente na célula 10.


In [1]:
# 1. GPU e dependências do runtime.
import torch, sys, subprocess
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab antes de continuar'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cryptography', 'huggingface_hub', 'pyarrow', 'scikit-learn'])
print('Dependências disponíveis.', flush=True)

GPU: Tesla T4
Dependências disponíveis.


In [2]:
# 2. Drive e pastas novas; não altera experimentos antigos.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for name in ('credenciais', 'dados_fase3_7', 'modelos', 'resultados'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| livre reportado:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive: /content/drive/MyDrive/TCC_Final | livre reportado: 5.6 GB
/content livre: 249.06 GB


In [3]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

Credencial desbloqueada nesta sessão.


In [4]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'fase3-ajuste-fino'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase37-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/fase3_7_dados.py', 'experimentos/fase3_7_finetune.py', 'experimentos/avaliar_fase3_7_mirage.py', 'Notebooks/fase_3/fase3.3_compressao.ipynb')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc37_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro os arquivos da Fase 3.7 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

Baixando snapshot do GitHub...
Código sincronizado: 77b84398a869b28f196d4799a5ef12578ce2ecbd


In [5]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
DATA = CODE_ROOT / 'experimentos/fase3_7_dados.py'
TRAIN = CODE_ROOT / 'experimentos/fase3_7_finetune.py'
for path in (DATA, TRAIN):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)

Código: /content/tcc_codigo/77b84398a869b28f196d4799a5ef12578ce2ecbd


In [6]:
# 6. PILOTO MANUAL: baixa dois shards completos em /content e confere classes e grupos.
SHARDS = ['data/HFCF_small_1.parquet', 'data/HFCF_small_118.parquet']
shard_args = [item for shard in SHARDS for item in ('--shard', shard)]
run_live('piloto 3.7', [sys.executable, '-u', str(DATA), 'pilot', '--root', str(ROOT), *shard_args])


piloto 3.7: iniciando
data/HFCF_small_1.parquet: candidatos reais=0 sintéticos=2927
data/HFCF_small_118.parquet: candidatos reais=2992 sintéticos=0
Aviso: reais com fonte n/a; divisão por imagem, não por fonte.
Piloto: {0: 2992, 1: 2927} | grupos reais= 1 | geradores= 76 | bytes= 1360897640 | commit= 6c539a534c07917307c381f5af4053c6091b5278
Classe 0: grupos elegíveis (20+): 1; principais: [('n/a', 2992)]
Classe 1: grupos elegíveis (20+): 76; principais: [('iamgokul/my-pet-dog-ggs', 51), ('songhee/rugged-sporty-car', 51), ('subhamoy12/my-pet-cat-xzr', 51), ('albe/textual_inversion_halpern', 41), ('andreamaduzzi/textual_inversion_cat', 41), ('ares1123/virtual-dress-try-on', 41), ('artificialguybr/iconsmi-appiconsmodelforsd', 41), ('aviiax/ds7', 41), ('botp/realistic_vision_v1.4', 41), ('coder119/vectorartz_diffusion', 41), ('coreco/seek.art_mega', 41), ('daniil-plotnikov/deepvision-v6-0', 41)]
Seleção possível por divisão/classe: {('train', 0): 1050, ('train', 1): 868, ('validation_calib

In [7]:
# 7. PREPARO MANUAL: congela selecao e divisoes antes de qualquer treino.
# Se o piloto indicar classes insuficientes, pare e informe a saida.
run_live('preparacao 3.7', [sys.executable, '-u', str(DATA), 'prepare', '--root', str(ROOT), *shard_args])


preparacao 3.7: iniciando
data/HFCF_small_1.parquet: candidatos reais=0 sintéticos=2927
data/HFCF_small_118.parquet: candidatos reais=2992 sintéticos=0
Aviso: reais com fonte n/a; divisão por imagem, não por fonte.
Seleção persistida: /content/drive/MyDrive/TCC_Final/dados_fase3_7/fase3_7_cfsmall_expansao_v1.incompleto/plan.json
preparacao 3.7: ativo há 0.6 min; sem nova saída
preparacao 3.7: ativo há 1.1 min; sem nova saída
preparacao 3.7: ativo há 1.6 min; sem nova saída
preparacao 3.7: ativo há 2.1 min; sem nova saída
Hashes OpenFake train: 500/2032
preparacao 3.7: ativo há 3.0 min; sem nova saída
preparacao 3.7: ativo há 3.5 min; sem nova saída
preparacao 3.7: ativo há 4.0 min; sem nova saída
preparacao 3.7: ativo há 4.5 min; sem nova saída
Hashes OpenFake train: 1000/2032
preparacao 3.7: ativo há 5.4 min; sem nova saída
preparacao 3.7: ativo há 5.9 min; sem nova saída
preparacao 3.7: ativo há 6.4 min; sem nova saída
preparacao 3.7: ativo há 6.9 min; sem nova saída
Hashes OpenFake 

In [8]:
# 8. COPIA LOCAL: execute em cada sessao e confira hashes de ambos os conjuntos.
OLD_DATA = CODE_ROOT / 'experimentos/fase3_5_dados.py'
run_live('copia local 3.5', [sys.executable, '-u', str(OLD_DATA), 'stage', '--root', str(ROOT)])
run_live('copia local 3.7', [sys.executable, '-u', str(DATA), 'stage', '--root', str(ROOT)])


copia local 3.5: iniciando
Cópia/verificação: 100/3000
Cópia/verificação: 200/3000
Cópia/verificação: 300/3000
Cópia/verificação: 400/3000
Cópia/verificação: 500/3000
Cópia/verificação: 600/3000
Cópia/verificação: 700/3000
Cópia/verificação: 800/3000
Cópia/verificação: 900/3000
Cópia/verificação: 1000/3000
Cópia/verificação: 1100/3000
Cópia/verificação: 1200/3000
Cópia/verificação: 1300/3000
Cópia/verificação: 1400/3000
Cópia/verificação: 1500/3000
Cópia/verificação: 1600/3000
Cópia/verificação: 1700/3000
Cópia/verificação: 1800/3000
Cópia/verificação: 1900/3000
Cópia/verificação: 2000/3000
Cópia/verificação: 2100/3000
Cópia/verificação: 2200/3000
Cópia/verificação: 2300/3000
Cópia/verificação: 2400/3000
Cópia/verificação: 2500/3000
Cópia/verificação: 2600/3000
Cópia/verificação: 2700/3000
Cópia/verificação: 2800/3000
Cópia/verificação: 2900/3000
Cópia/verificação: 3000/3000
Dados internos locais prontos: /content/fase3_5_novos_v1
copia local 3.5: concluído em 10.0 min
copia local 3.7:

In [9]:
# 9. SMOKE MANUAL: divisoes, GPU, checkpoint, forward e gravacao; sem treino.
run_live('smoke 3.7', [sys.executable, '-u', str(TRAIN), 'smoke'])


smoke 3.7: iniciando
GPU: Tesla T4
Arquitetura da Fase 2 montada | ramos: ['vit', 'fft', 'cnn'] | fusão: gate | T2=0.391 | limiar2=0.725 | parâmetros: 10.06M
Partições 3.7: 5512 1260 1260 | teste histórico reservado, não avaliado: 1947
Hashes treino: 500/5512
Hashes treino: 1000/5512
Hashes treino: 1500/5512
Hashes treino: 2000/5512
Hashes treino: 2500/5512
Hashes treino: 3000/5512
Hashes treino: 3500/5512
Hashes treino: 4000/5512
Hashes treino: 4500/5512
Hashes treino: 5000/5512
Hashes treino: 5500/5512
Hashes seleção: 500/1260
Hashes seleção: 1000/1260
Hashes calibração: 500/1260
Hashes calibração: 1000/1260
Hashes teste reservado: 500/1947
Hashes teste reservado: 1000/1947
Hashes teste reservado: 1500/1947
Pesos iniciais da Fase 3.3 carregados: /content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/phase3_final.pt
Smoke OK: /content/drive/MyDrive/TCC_Final/modelos/fase3_7_cfsmall_expansao_v1_smoke/smoke.json
smoke 3.7: concluído em 0.4 min


In [10]:
# 10. TREINO MANUAL: inicia ou retoma do ultimo last.pt.
# Execute somente apos conferir Smoke OK na celula 9.
run_live('ajuste fino 3.7', [sys.executable, '-u', str(TRAIN), 'auto'])


ajuste fino 3.7: iniciando
GPU: Tesla T4
Arquitetura da Fase 2 montada | ramos: ['vit', 'fft', 'cnn'] | fusão: gate | T2=0.391 | limiar2=0.725 | parâmetros: 10.06M
Partições 3.7: 5512 1260 1260 | teste histórico reservado, não avaliado: 1947
Hashes treino: 500/5512
Hashes treino: 1000/5512
Hashes treino: 1500/5512
Hashes treino: 2000/5512
Hashes treino: 2500/5512
Hashes treino: 3000/5512
Hashes treino: 3500/5512
Hashes treino: 4000/5512
Hashes treino: 4500/5512
Hashes treino: 5000/5512
Hashes treino: 5500/5512
Hashes seleção: 500/1260
Hashes seleção: 1000/1260
Hashes calibração: 500/1260
Hashes calibração: 1000/1260
Hashes teste reservado: 500/1947
Hashes teste reservado: 1000/1947
Hashes teste reservado: 1500/1947
Pesos iniciais da Fase 3.3 carregados: /content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/phase3_final.pt
ajuste fino 3.7: ativo há 0.7 min; sem nova saída
[E01/12] lr=3.0e-05 | train_loss=0.4913 | val_loss=0.3762 | AUC=0.9254 | bal_acc=0.8413 | 54s  -> melhor
aj

In [11]:
# 11. LEITURA INTERNA: selecao escolhe epoca; calibracao define T e limiar.
run_dir = ROOT / 'modelos/fase3_7_cfsmall_expansao_v1'
for name in ('experiment_config.json', 'split_manifest.json', 'history.csv', 'last.pt',
             'best.pt', 'phase3_final.pt', 'internal_metrics.json', 'training_complete.json'):
    print(name, 'presente' if (run_dir / name).is_file() else 'AUSENTE')
metrics = run_dir / 'internal_metrics.json'
print(json.dumps(json.loads(metrics.read_text(encoding='utf-8')), indent=2)
      if metrics.is_file() else 'Treino interno ainda nao concluido.')


experiment_config.json presente
split_manifest.json presente
history.csv presente
last.pt presente
best.pt presente
phase3_final.pt presente
internal_metrics.json presente
training_complete.json presente
{
  "best_epoch": 11,
  "selection_auc": 0.9699539052496798,
  "calibration_bal_acc_apparent": 0.9323559539052497,
  "temperature": 0.4599452316761017,
  "threshold": 0.3449999999999999,
  "train_count": 5512,
  "selection_count": 1260,
  "calibration_count": 1260,
  "external_used": false,
  "openfake_test_used": false
}


## Comparação externa após o treino

As células 12–14 executam o protocolo Mirage de 2.000 imagens congeladas. A célula 13 registra hashes de pesos, código e manifesto antes de inferir. Temperatura e limiar vêm da calibração interna. O Mirage já foi observado e não é um teste cego independente para esta variante. Uma futura avaliação final independente exige outro conjunto ainda não observado.


In [12]:
# 12. MIRAGE: copia local verificada, sem novo download da fonte.
MIRAGE_DATA = CODE_ROOT / 'experimentos/benchmark_mirage_2000.py'
run_live('copia local Mirage', [sys.executable, '-u', str(MIRAGE_DATA), 'stage', '--root', str(ROOT)])


copia local Mirage: iniciando
Cópia/verificação: 100/2000
Cópia/verificação: 200/2000
Cópia/verificação: 300/2000
Cópia/verificação: 400/2000
Cópia/verificação: 500/2000
Cópia/verificação: 600/2000
Cópia/verificação: 700/2000
Cópia/verificação: 800/2000
Cópia/verificação: 900/2000
Cópia/verificação: 1000/2000
Cópia/verificação: 1100/2000
Cópia/verificação: 1200/2000
Cópia/verificação: 1300/2000
Cópia/verificação: 1400/2000
Cópia/verificação: 1500/2000
Cópia/verificação: 1600/2000
Cópia/verificação: 1700/2000
Cópia/verificação: 1800/2000
Cópia/verificação: 1900/2000
Cópia/verificação: 2000/2000
Benchmark local pronto: /content/benchmark_externo_mirage_2000_v1
copia local Mirage: concluído em 0.2 min


In [13]:
# 13. CONGELAMENTO: checkpoint, temperatura, limiar, codigo e manifesto.
EVAL = CODE_ROOT / 'experimentos/avaliar_fase3_7_mirage.py'
run_live('congelamento 3.7', [sys.executable, '-u', str(EVAL), 'freeze', '--root', str(ROOT)])


congelamento 3.7: iniciando
Configuração 3.7 congelada antes da inferência: /content/drive/MyDrive/TCC_Final/resultados/benchmark_externo_mirage_2000_v1/locks/fase3_7_cfsmall_expansao_v1.json
{
  "model": "fase3_7_cfsmall_expansao_v1",
  "checkpoint_sha256": "cf3c9305dda7b01d05cef1520437470208b6cb3a8d45a37b4c5bdecd38215062",
  "config_sha256": "2799beff902347b518dcf267bdd383f0d00640f7b82a97f7389a1c5be9702d88",
  "internal_metrics_sha256": "793fd590af46bdb06c2911e0571e7433c31861fcd000b729f847db62341e254d",
  "architecture_source_sha256": "3db23c93de1130c4b55a1b1ad2f3f522005e5e2dd3d15733698f1f2418eb7c7b",
  "evaluation_script_sha256": "4e595d4fb14ceb8bcf4094fda09ee6144f108c288287e8c64b49b33836a614a5",
  "manifest_sha256": "4b3658f880e022957557933fd662c69485c19a5ed9a04e2e09c32ce485737190",
  "prior_metrics_sha256": "0f88e95ed7458059eb8dc09438d64aac0a20cdea399d3a6ff88b16f38d0a057b",
  "prior_predictions_sha256": "ec6a12597b23a82e219c2e417dbb72ee86d5330fb3cbe36815ad368704bbcbb2",
  "tempera

In [14]:
# 14. AVALIACAO EXTERNA COMPARATIVA apos o congelamento.
run_live('avaliacao Mirage 3.7', [sys.executable, '-u', str(EVAL), 'evaluate', '--root', str(ROOT)])


avaliacao Mirage 3.7: iniciando
Integridade antes da inferência: 100/2000
Integridade antes da inferência: 200/2000
Integridade antes da inferência: 300/2000
Integridade antes da inferência: 400/2000
Integridade antes da inferência: 500/2000
Integridade antes da inferência: 600/2000
Integridade antes da inferência: 700/2000
Integridade antes da inferência: 800/2000
Integridade antes da inferência: 900/2000
Integridade antes da inferência: 1000/2000
Integridade antes da inferência: 1100/2000
Integridade antes da inferência: 1200/2000
Integridade antes da inferência: 1300/2000
Integridade antes da inferência: 1400/2000
Integridade antes da inferência: 1500/2000
Integridade antes da inferência: 1600/2000
Integridade antes da inferência: 1700/2000
Integridade antes da inferência: 1800/2000
Integridade antes da inferência: 1900/2000
Integridade antes da inferência: 2000/2000
Fase 3.7 congelada | GPU=Tesla T4 | T=0.459945 | limiar=0.345000
Inferência externa: 100/2000
Inferência externa: 200

In [15]:
# 15. LEITURA do resultado preservado; sem recalibracao.
result = ROOT / 'resultados/benchmark_externo_mirage_2000_v1/fase3_7_cfsmall_expansao_v1/metrics.json'
print(json.dumps(json.loads(result.read_text(encoding='utf-8')), indent=2)
      if result.is_file() else 'Avaliacao externa ainda nao executada.')


{
  "model": "fase3_7_cfsmall_expansao_v1",
  "n": 2000,
  "real": 1000,
  "fake": 1000,
  "accuracy": 0.623,
  "accuracy_ci95_wilson": [
    0.6015435902777672,
    0.6439848160910445
  ],
  "balanced_accuracy": 0.623,
  "auc": 0.6908315000000002,
  "precision": 0.694006309148265,
  "recall_fake": 0.44,
  "specificity_real": 0.806,
  "f1": 0.5385556915544675,
  "tn": 806,
  "fp": 194,
  "fn": 560,
  "tp": 440,
  "temperature_frozen": 0.4599452316761017,
  "threshold_frozen": 0.3449999999999999,
  "upscaled_short_side_below_224": 0,
  "checkpoint_sha256": "cf3c9305dda7b01d05cef1520437470208b6cb3a8d45a37b4c5bdecd38215062",
  "manifest_sha256": "4b3658f880e022957557933fd662c69485c19a5ed9a04e2e09c32ce485737190",
  "evaluation_script_sha256": "4e595d4fb14ceb8bcf4094fda09ee6144f108c288287e8c64b49b33836a614a5",
  "architecture_source_sha256": "3db23c93de1130c4b55a1b1ad2f3f522005e5e2dd3d15733698f1f2418eb7c7b",
  "architecture_extraction_sha256": "ad5db5ee692cd67bc3838ff814b36a179f3eb1acc2cdd7